# Week 11 — Privacy: re-identification, k-anonymity, and a *statable* guarantee

**Lesson plan:** [`../weeks/week-11.md`](../weeks/week-11.md)
**Reading:** Narayanan & Shmatikov (2008), *Robust De-anonymization of Large Sparse
Datasets* (the Netflix Prize paper).

Pure Python — no lab target, no model required. Synthetic data throughout, so
nothing here is real personal data (which is itself the week's ethics rule).

> ### The one idea
> "Anonymized" almost never means anonymous. Dropping the name does nothing if the
> **quasi-identifiers** that remain are unique. And the fixes divide sharply:
> k-anonymity and its kin **raise the attacker's cost with no guarantee**, while
> **differential privacy gives a guarantee you can state as a number (ε)** — the
> first such control in the whole course. Knowing which kind of defense you have is
> the week's skill.

## 1 · Re-identification by linkage — the Sweeney attack

A hospital releases "anonymized" records: names removed, but ZIP, year of birth,
and sex kept "because those aren't identifying." Latanya Sweeney's result: the
triple **{ZIP, birthdate, sex} uniquely identifies ~87% of Americans.** Cross it
against any public dataset that has names plus those fields (a voter roll, a leak),
and the anonymization evaporates.

We build a synthetic release and a synthetic public "voter roll," then link them.

In [1]:
import random
from collections import Counter, defaultdict

SEED = 20250807
rng = random.Random(SEED)

ZIPS = ["02101", "02102", "02139"]
SEX = ["M", "F"]
DIAGNOSES = ["flu", "diabetes", "hiv", "cancer", "none"]

# The true population (we know everything because we generated it).
population = []
for i in range(120):
    population.append({
        "name": f"person{i:03d}",
        "zip": rng.choice(ZIPS),
        "yob": rng.randint(1950, 2000),
        "sex": rng.choice(SEX),
        "diagnosis": rng.choice(DIAGNOSES),
    })

# The RELEASED medical data: name dropped, everything else kept.
released = [{k: v for k, v in p.items() if k != "name"} for p in population]

# The AUXILIARY public data (a voter roll): name + the quasi-identifiers.
voter_roll = [{"name": p["name"], "zip": p["zip"], "yob": p["yob"], "sex": p["sex"]}
              for p in population]

print(f"released {len(released)} 'anonymized' medical records (no names)")
print("example:", released[0])

released 120 'anonymized' medical records (no names)
example: {'zip': '02139', 'yob': 1952, 'sex': 'M', 'diagnosis': 'diabetes'}


In [7]:
def quasi_id(r):
    """The fields that were kept 'because they aren't identifying'."""
    return (r["zip"], r["yob"], r["sex"])

# How many released records are UNIQUE on the quasi-identifier?
counts = Counter(quasi_id(r) for r in released)
unique = [r for r in released if counts[quasi_id(r)] == 1]
print(f"uniquely identified by (zip, yob, sex): {len(unique)}/{len(released)} "
      f"({len(unique)/len(released):.0%})\n")

# LINK a unique record back to a name via the voter roll. This is the oracle: a
# re-identification is CONFIRMED when we can name the person behind a medical row.
name_by_qid = {quasi_id(v): v["name"] for v in voter_roll}
victim = unique[0]
print("Attack — take one 'anonymous' medical record:")
print(f"  {victim}")
print(f"  its quasi-id {quasi_id(victim)} appears once in the voter roll ->")
print(f"  RE-IDENTIFIED as {name_by_qid[quasi_id(victim)]}, "
      f"with diagnosis {victim['diagnosis']!r}")

uniquely identified by (zip, yob, sex): 90/120 (75%)

Attack — take one 'anonymous' medical record:
  {'zip': '02101', 'yob': 1964, 'sex': 'F', 'diagnosis': 'diabetes'}
  its quasi-id ('02101', 1964, 'F') appears once in the voter roll ->
  RE-IDENTIFIED as person001, with diagnosis 'diabetes'


### The lesson

Nobody "hacked" anything. The data was released exactly as policy required — name
removed. The attack is pure **linkage**: the quasi-identifiers that felt harmless
are a fingerprint. This is week 1's move (name the assumption: *"these fields
aren't identifying"* — false) applied to privacy.

> Any field an attacker can also learn from *another* source is a quasi-identifier.
> That set is almost always bigger than the data owner thinks.

## 2 · k-anonymity — generalize until you hide in a crowd

The classic fix: **generalize** the quasi-identifiers (ZIP → prefix, birth year →
decade) until every combination is shared by at least *k* people. Then no single
record can be picked out of its group.

In [8]:
def generalize(r):
    """Coarsen the quasi-identifiers: 3-digit ZIP prefix, 20-year birth band."""
    return (r["zip"][:3], (r["yob"] // 20) * 20)

groups = defaultdict(list)
for r in released:
    groups[generalize(r)].append(r)

k = min(len(v) for v in groups.values())
print(f"after generalization: {len(groups)} groups; smallest group k = {k}")
for g, members in groups.items():
    print(f"  group {g}: {len(members)} people, "
          f"{len(set(m['diagnosis'] for m in members))} distinct diagnoses")
print("\nNo record is unique now — identity is hidden in a crowd of at least", k, ".")

after generalization: 4 groups; smallest group k = 3
  group ('021', 1940): 23 people, 5 distinct diagnoses
  group ('021', 1960): 47 people, 5 distinct diagnoses
  group ('021', 1980): 47 people, 5 distinct diagnoses
  group ('021', 2000): 3 people, 3 distinct diagnoses

No record is unique now — identity is hidden in a crowd of at least 3 .


## 3 · ⚠️ ...but k-anonymity protects *identity*, not the *secret*

k-anonymity says nothing about the **sensitive attribute**. If everyone in a group
shares the same diagnosis, you learn it *without* singling anyone out — the
**homogeneity attack**, and the reason *l*-diversity was proposed (require ≥ *l*
distinct sensitive values per group).

Here is the teaching case, constructed so the failure is unmistakable:

In [4]:
# A k-anonymous group (k=5, nobody is unique) where the sensitive value is uniform.
homogeneous_group = [
    {"zip": "021xx", "yob": "1960s", "diagnosis": "hiv"} for _ in range(5)
]
distinct = set(m["diagnosis"] for m in homogeneous_group)

print(f"group of {len(homogeneous_group)} people, all with generalized qid "
      f"(021xx, 1960s)  -> k-anonymous, k={len(homogeneous_group)}")
print(f"distinct diagnoses in the group: {distinct}")
print()
print("An attacker who knows their victim is a 1960s-born 021xx resident cannot")
print("say WHICH row is the victim — identity is protected. But every row has the")
print(f"same diagnosis, so they learn it anyway: {list(distinct)[0]!r}.")
print()
print("k-anonymity: PROTECTS IDENTITY, NOT THE ATTRIBUTE. No guarantee about what")
print("an attacker learns. This is why it 'raises cost' rather than 'guarantees'.")

group of 5 people, all with generalized qid (021xx, 1960s)  -> k-anonymous, k=5
distinct diagnoses in the group: {'hiv'}

An attacker who knows their victim is a 1960s-born 021xx resident cannot
say WHICH row is the victim — identity is protected. But every row has the
same diagnosis, so they learn it anyway: 'hiv'.

k-anonymity: PROTECTS IDENTITY, NOT THE ATTRIBUTE. No guarantee about what
an attacker learns. This is why it 'raises cost' rather than 'guarantees'.


## 4 · Differential privacy — a guarantee you can state as a number

Everything so far has been a cost-raiser with a workaround. **Differential privacy
is different.** It answers a query with calibrated random noise such that the output
is *almost identical* whether or not any one person is in the dataset. Formally, for
privacy budget ε, adding or removing one person changes the probability of any
output by at most a factor of e^ε.

That is a **guarantee** — the scorecard's axis 2, stated as a conditional you can
actually write down:

> *No adversary, with any side information, can determine whether a given individual
> is in the dataset with advantage beyond e^ε — regardless of what they know.*

The cost is accuracy. Watch the tradeoff on a simple count query.

In [5]:
import math

def laplace_noise(scale, r):
    """Sample from a Laplace distribution (the DP mechanism for counts)."""
    u = r.uniform(-0.5, 0.5)
    return -scale * math.copysign(1, u) * math.log(1 - 2 * abs(u))

true_count = sum(1 for r in released if r["diagnosis"] == "hiv")
print(f"true answer to COUNT(diagnosis = hiv): {true_count}\n")

# Sensitivity of a count query is 1 (one person changes it by at most 1).
# Laplace noise scale = sensitivity / epsilon.
print(f"  {'epsilon':>8}{'noise scale':>13}{'mean |error|':>14}{'privacy':>14}")
print("  " + "-" * 49)
for eps in (0.1, 0.5, 1.0, 5.0):
    scale = 1.0 / eps
    r = random.Random(1)
    errors = [abs(laplace_noise(scale, r)) for _ in range(5000)]
    label = {0.1: "very strong", 0.5: "strong", 1.0: "moderate", 5.0: "weak"}[eps]
    print(f"  {eps:>8}{scale:>13.1f}{sum(errors)/len(errors):>14.2f}{label:>14}")

print("""
Small epsilon -> more noise -> stronger privacy but a less accurate answer.
Large epsilon -> less noise -> accurate but weak privacy.

THIS is the control with a knob AND a guarantee. Contrast week 9: no system-prompt
or output filter could give you a sentence like the axis-2 statement above. DP can.""")

true answer to COUNT(diagnosis = hiv): 24

   epsilon  noise scale  mean |error|       privacy
  -------------------------------------------------
       0.1         10.0         10.17   very strong
       0.5          2.0          2.03        strong
       1.0          1.0          1.02      moderate
       5.0          0.2          0.20          weak

Small epsilon -> more noise -> stronger privacy but a less accurate answer.
Large epsilon -> less noise -> accurate but weak privacy.

THIS is the control with a knob AND a guarantee. Contrast week 9: no system-prompt
or output filter could give you a sentence like the axis-2 statement above. DP can.


## 5 · Membership inference — privacy attacks reach *models*, not just tables

The dataset isn't the only thing that leaks. A model **trained** on personal data
can leak who was in its training set. **Membership inference:** given a record and
access to the model, decide whether that record was used to train it. The tell is
overfitting — a model is systematically more confident (lower loss) on data it
memorized than on data it never saw.

We simulate a model's per-record loss (low for training members, higher for
non-members, with realistic overlap) and mount the attack.

In [6]:
r = random.Random(SEED)

def cohort(n, is_member):
    # Members were memorized -> lower loss; non-members -> higher; distributions overlap.
    mu = 0.3 if is_member else 0.9
    return [{"member": is_member, "loss": r.gauss(mu, 0.35)} for _ in range(n)]

probe = cohort(100, True) + cohort(100, False)   # 50/50, so random guessing = 50%

def infer(threshold):
    """Guess 'member' when loss < threshold (the model was confident -> it saw it)."""
    correct = sum(1 for d in probe if (d["loss"] < threshold) == d["member"])
    tp = sum(1 for d in probe if d["member"] and d["loss"] < threshold)
    fp = sum(1 for d in probe if not d["member"] and d["loss"] < threshold)
    prec = tp / (tp + fp) if tp + fp else 0
    return correct / len(probe), prec

print(f"  {'threshold':>10}{'accuracy':>10}{'precision':>11}")
print("  " + "-" * 31)
for t in (0.4, 0.6, 0.8):
    acc, prec = infer(t)
    print(f"  {t:>10}{acc:>10.0%}{prec:>11.0%}")

print("""
Random guessing would be 50% accuracy. Beating it means the MODEL LEAKED whether a
record was in training — a privacy breach with no table involved. If those records
are patients, being in the 'diabetes model's' training set IS the sensitive fact.

The defense is the same guarantee from section 4: training with differential
privacy (DP-SGD) bounds how much any one training record can move the model, which
bounds membership inference by the same epsilon.""")

   threshold  accuracy  precision
  -------------------------------
         0.4       78%        90%
         0.6       80%        80%
         0.8       76%        71%

Random guessing would be 50% accuracy. Beating it means the MODEL LEAKED whether a
record was in training — a privacy breach with no table involved. If those records
are patients, being in the 'diabetes model's' training set IS the sensitive fact.

The defense is the same guarantee from section 4: training with differential
privacy (DP-SGD) bounds how much any one training record can move the model, which
bounds membership inference by the same epsilon.


## 6 · The scorecard view — two kinds of privacy defense

| Defense | What it does | Guarantee (axis 2)? | Its failure |
|---|---|---|---|
| Drop the name | removes one identifier | **none** | linkage on quasi-identifiers |
| k-anonymity | generalize to hide in a crowd of k | **none** — protects identity, not the attribute | homogeneity / background knowledge |
| l-diversity | + require diverse sensitive values | weak | skewed / semantic closeness |
| **Differential privacy** | calibrated noise, budget ε | **yes — e^ε bound, against any side information** | accuracy cost; budget composes across queries |

> The through-line of the course: most controls **raise cost**; a rare few give a
> **statable guarantee**. Week 9 had none (prompt-injection filters). Week 10 had
> one (least privilege, enforced in code). Week 11 has the strongest: **differential
> privacy, a guarantee stated as a number.** Learn to tell them apart — it is the
> difference between "we think this is safer" and "here is what we can prove."

## 7 · Your studio deliverable

In `week11/`:

1. **Re-identify** — link the released dataset to the voter roll; report the % of
   records uniquely identified and name one. The oracle is a successful name link.
2. **k-anonymize, then break it** — generalize to some k, then find (or construct) a
   homogeneity attack that learns a sensitive value without singling anyone out.
3. **Differential privacy** — add Laplace noise to an aggregate query at ≥ 4 epsilon
   values; plot accuracy vs. epsilon and state the guarantee each ε gives.
4. **Membership inference** — mount the loss-threshold attack; report accuracy above
   the 50% baseline.
5. **Control Scorecard** — for each defense, state whether it gives a *guarantee*
   or only *raises cost*, and back it with your numbers. This distinction is the
   whole grade.

> **Duel 3 (AI red team) is due this week, and capstone peer-red-team assignments
> are made.** Bring the same measurement rigor to your peer's app that you brought
> to re-identifying a dataset here.